# Customer Segmentation in Python

## Project Overview

In this project, I explore customer segmentation using transactional retail data.  
The goal is to group customers into meaningful segments based on their purchasing behavior.

This notebook covers the **first stage** of the project:

- loading the dataset
- understanding its structure
- performing an initial inspection
- identifying potential data quality issues

Later steps will include:

- data cleaning
- customer-level feature engineering
- scaling
- KMeans clustering
- cluster evaluation
- business interpretation

## 1. Importing the Required Libraries

Before working with the data, I import the main Python libraries needed for analysis.

### Why these libraries?

- **pandas**: for working with tabular data
- **numpy**: for numerical operations
- **matplotlib**: for visualizations
- **Path** from `pathlib`: for clean and readable file path handling

Using `Path` is considered a good practice because it makes file paths easier to manage across the project structure.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

## 2. Notebook Display Settings

These settings improve readability inside the notebook.

### What do they do?

- show more columns when displaying a DataFrame
- make table outputs easier to read
- set a default figure size for charts

These are convenience settings only.  
They do not change the dataset itself.

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

plt.rcParams["figure.figsize"] = (10, 6)

## 3. Defining Project Paths

This project uses a structured folder layout.  
Instead of hardcoding long file paths everywhere, I define the main paths once and reuse them later.

### Why is this useful?

It makes the notebook:

- cleaner
- easier to maintain
- easier to reuse later

It also fits a more professional project workflow, especially for GitHub portfolio projects.

In [4]:
PROJECT_ROOT = Path.cwd().parent
DATA_RAW_PATH = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
IMAGES_PATH = PROJECT_ROOT / "images"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", DATA_RAW_PATH)

Project root: C:\Projects\customer-segmentation-python
Raw data path: C:\Projects\customer-segmentation-python\data\raw


## 4. Loading the Dataset

The dataset used in this project is the **Online Retail** dataset.  
It contains transactional data from an online retail store.

### Important note

This is **transaction-level data**, not customer-level data.

That means:

- one customer can appear multiple times
- one invoice can contain multiple products
- customer segmentation cannot be done directly on the raw table

Later, I will transform this data into a **customer-level dataset** suitable for clustering.

In [6]:
file_path = DATA_RAW_PATH / "Online Retail.xlsx"
df = pd.read_excel(file_path)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 5. First Look at the Data

The first few rows help verify that the file loaded correctly and give an initial idea of the dataset structure.

At this point, I am mainly checking:

- column names
- whether the data looks consistent
- whether the values seem reasonable
- whether the dataset includes customer-related identifiers

## 6. Dataset Shape and Columns

Now I check:

- how many rows and columns the dataset has
- the names of the columns

This is a basic but essential first step in every data analysis project.

In [7]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (541909, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


## 7. Data Types and General Structure

Using `df.info()` helps understand:

- the data types of each column
- whether columns are numeric, text, or datetime
- how many non-null values are present

This is especially important because later steps such as feature engineering and clustering require well-structured numeric inputs.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


## 8. Missing Values

Missing values often indicate data quality issues or incomplete records.

At this stage, I want to identify:

- which columns contain missing values
- how serious the missingness is
- whether key fields such as `CustomerID` are affected

This is important because customer segmentation requires reliable customer-level information.

In [9]:
df.isna().sum().sort_values(ascending=False)

CustomerID     135080
Description      1454
StockCode           0
InvoiceNo           0
Quantity            0
InvoiceDate         0
UnitPrice           0
Country             0
dtype: int64

## 9. Inspecting the First 10 Rows

Looking at a slightly larger sample helps identify patterns in the raw transactional data.

For example:

- repeated invoice numbers
- repeated customer IDs
- product-level rows
- descriptions and prices

In [10]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


## 10. Descriptive Statistics

Descriptive statistics provide a quick overview of the data distribution.

For numeric variables, they show values such as:

- mean
- standard deviation
- minimum
- maximum
- quartiles

This helps detect unusual values early, such as:

- negative quantities
- zero prices
- strong skewness
- extreme outliers

In [11]:
df.describe(include="all")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909.0,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900.0,4070,4223,NaN,NaN,NaN,NaN,38
top,573585.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1114.0,2313,2369,NaN,NaN,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570,NaN
min,NaN,NaN,NaN,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,2011-03-28 11:34:00,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,2011-07-19 17:17:00,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,2011-10-19 11:27:00,4.130000,16791.000000,NaN
max,NaN,NaN,NaN,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000,NaN


## 11. Initial Observations

At this stage, the goal is not yet to clean or model the data, but to understand its structure.

Based on this initial inspection, I expect the following kinds of issues:

- missing values in customer-related fields
- repeated customer entries because the data is transactional
- potential cancellations or returns
- outliers in quantity or price
- the need to aggregate data from transaction level to customer level

### Why this matters for segmentation

Customer segmentation requires one row per customer, not one row per purchased item.  
This means that a key next step will be transforming the raw dataset into meaningful customer-level features.